# Experiment 2: linear CKA across LLaVA vision layers

Images are observations. For each image and returned hidden-state index, the complete `[577, 1024]` token tensor is reduced to one fixed 1024-dimensional feature vector by taking the arithmetic mean over **all** tokens, including the CLS token. Mean pooling is deterministic, retains every token, and keeps the in-memory feature array small; flattening would create roughly 590,848 features per image per layer.

The CKA calculation follows Google's official [`representation_similarity/Demo.ipynb`](https://github.com/google-research/google-research/blob/master/representation_similarity/Demo.ipynb): form linear Gram matrices, double-center them, and normalize their Hilbert–Schmidt inner product. The diagonal is therefore one (up to floating-point precision).

Run from this notebook's directory. Install visualization/dataframe dependencies in the existing AMP LLaVA environment if needed:

```bash
uv pip install pandas matplotlib
```


In [ ]:
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from transformers import LlavaForConditionalGeneration

MODEL_ID = "llava-hf/llava-1.5-7b-hf"
DTYPE = torch.float16
DEVICE = torch.device("cuda")
DATASET_DIR = Path("../../dataset/laion_art/clean")
N_IMAGES = 256
RANDOM_SEED = 2025
BATCH_SIZE = 4
NUM_WORKERS = 4
OUTPUT_CSV = Path("exp2_linear_cka.csv")
IMAGE_SUFFIXES = {".png"}

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required, matching AMP and Exp1.")


In [ ]:
def select_images(directory, count, seed):
    paths = sorted(path for path in directory.iterdir() if path.suffix.lower() in IMAGE_SUFFIXES)
    if len(paths) < count:
        raise ValueError(f"Requested {count} images, but only found {len(paths)} in {directory}")
    return random.Random(seed).sample(paths, count)


class ImagePathDataset(Dataset):
    def __init__(self, paths, transform):
        self.paths = paths
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        # Images prepared by prepare_laion_art.ipynb are RGB PNGs.
        with Image.open(self.paths[index]) as image:
            return self.transform(image.convert("RGB"))


# Exactly the FP16, resize, and normalization conventions used in Exp1/AMP.
preprocess = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Resize(
            (336, 336),
            interpolation=transforms.InterpolationMode.BICUBIC,
        ),
        transforms.Normalize(
            (0.48145466, 0.4578275, 0.40821073),
            (0.26862954, 0.26130258, 0.27577711),
        ),
    ]
)

image_paths = select_images(DATASET_DIR, N_IMAGES, RANDOM_SEED)
loader = DataLoader(
    ImagePathDataset(image_paths, preprocess),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
)


In [ ]:
llava_model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    low_cpu_mem_usage=True,
)
vision_tower = llava_model.vision_tower.to(DEVICE).eval()
del llava_model


def extract_layer_features(model, batches, device, dtype):
    """Return [layers, images, hidden_width] CPU features without saving intermediates."""
    chunks = None
    with torch.inference_mode():
        for images in batches:
            outputs = model(
                images.to(device=device, dtype=dtype, non_blocking=True),
                output_hidden_states=True,
            )
            pooled = [state.mean(dim=1).float().cpu() for state in outputs.hidden_states]
            if chunks is None:
                chunks = [[] for _ in pooled]
            if len(pooled) != len(chunks):
                raise RuntimeError("Vision tower returned a different number of layers")
            for layer_chunks, features in zip(chunks, pooled):
                layer_chunks.append(features)
    return torch.stack([torch.cat(layer_chunks, dim=0) for layer_chunks in chunks]).numpy()


features = extract_layer_features(vision_tower, loader, DEVICE, DTYPE)
num_states = features.shape[0]
assert num_states == vision_tower.config.num_hidden_layers + 1
layer_labels = ["Embedding", *map(str, range(1, num_states))]
print("features [layers, images, dimensions]:", features.shape)


In [ ]:
# Adapted directly from Google's public CKA demo formulation.
def center_gram(gram, unbiased=False):
    if not np.allclose(gram, gram.T):
        raise ValueError("Input must be a symmetric Gram matrix.")
    gram = gram.copy()
    if unbiased:
        np.fill_diagonal(gram, 0)
        means = np.sum(gram, axis=0, dtype=np.float64) / (gram.shape[0] - 2)
        means -= np.sum(means) / (2 * (gram.shape[0] - 1))
        gram -= means[:, None]
        gram -= means[None, :]
        np.fill_diagonal(gram, 0)
    else:
        means = np.mean(gram, axis=0, dtype=np.float64)
        means -= np.mean(means) / 2
        gram -= means[:, None]
        gram -= means[None, :]
    return gram


def cka(gram_x, gram_y, debiased=False):
    gram_x = center_gram(gram_x, unbiased=debiased)
    gram_y = center_gram(gram_y, unbiased=debiased)
    scaled_hsic = gram_x.ravel().dot(gram_y.ravel())
    normalization_x = np.linalg.norm(gram_x)
    normalization_y = np.linalg.norm(gram_y)
    return scaled_hsic / (normalization_x * normalization_y)


def linear_cka_matrix(layer_features, debiased=False):
    """Compute all layer pairs with images as Gram-matrix observations."""
    grams = [layer @ layer.T for layer in layer_features]
    count = len(grams)
    matrix = np.empty((count, count), dtype=np.float64)
    for row in range(count):
        for column in range(row, count):
            value = cka(grams[row], grams[column], debiased=debiased)
            matrix[row, column] = matrix[column, row] = value
    return matrix


cka_matrix = linear_cka_matrix(features, debiased=False)
cka_frame = pd.DataFrame(cka_matrix, index=layer_labels, columns=layer_labels)
cka_frame.to_csv(OUTPUT_CSV, index_label="layer")
cka_frame


In [ ]:
fig, axis = plt.subplots(figsize=(10, 8))
image = axis.imshow(cka_matrix, vmin=0, vmax=1, cmap="viridis", origin="upper")
axis.set_xticks(range(len(layer_labels)), labels=layer_labels, rotation=90)
axis.set_yticks(range(len(layer_labels)), labels=layer_labels)
axis.set_xlabel("Layer")
axis.set_ylabel("Layer")
axis.set_title(f"LLaVA-1.5-7B linear CKA ({N_IMAGES} LAION-Art images)")
fig.colorbar(image, ax=axis, label="Linear CKA")
fig.tight_layout()
plt.show()
